# Data preparation: from the Kaggle files to `fever_slice.csv`

This notebook shows where the knowledge base data comes from and every step used to get it.

- **Source:** Kaggle "Disease Prediction Using Machine Learning" (kaushil268). It has two files, a training file and a test file.
- **What we do:** merge the two files, clean them, keep 7 infectious diseases that cause fever, and add a category for each disease.
- **Result:** `fever_slice.csv`, the data used to build the knowledge base. The cleaning here is the same as in `prepare_data.py`.

Every number below is computed from the files when you run the notebook.

In [1]:
import os
import pandas as pd

TRAIN_INPUT = "data/Training.csv"
TEST_INPUT = "data/Testing.csv"
EXISTING_SLICE = "fever_slice.csv"     # only used for the final comparison

# Hand-written grouping. It is not in the original data.
CATEGORY = {
    "Malaria": "ParasiticInfection",
    "Dengue": "ViralInfection",
    "Chicken pox": "ViralInfection",
    "Common Cold": "ViralInfection",
    "Typhoid": "BacterialInfection",
    "Tuberculosis": "BacterialInfection",
    "Pneumonia": "BacterialInfection",
}

MIN_COUNT = 1      # keep a symptom if it appears at least this many times in the 7 diseases

pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 12)

## 1. The raw files

Each row is one patient case. Each symptom column is 1 (present) or 0 (absent). The column `prognosis` is the disease.

In [2]:
train = pd.read_csv(TRAIN_INPUT)
test = pd.read_csv(TEST_INPUT)

print(f"Training file: {train.shape[0]} rows, {train.shape[1]} columns")
print(f"Testing file:  {test.shape[0]} rows, {test.shape[1]} columns")
print(f"Diseases in training: {train['prognosis'].nunique()}")
print(f"Diseases in testing:  {test['prognosis'].nunique()}")

train_counts = train["prognosis"].value_counts()
test_counts = test["prognosis"].value_counts()
print(f"Rows per disease in training: min {train_counts.min()}, max {train_counts.max()}")
print(f"Rows per disease in testing:  min {test_counts.min()}, max {test_counts.max()}")

Training file: 4920 rows, 134 columns
Testing file:  42 rows, 133 columns
Diseases in training: 41
Diseases in testing:  41
Rows per disease in training: min 120, max 120
Rows per disease in testing:  min 1, max 2


In [3]:
# A few columns of the first rows (the file has too many columns to show all)
shown = list(train.columns[:4]) + list(train.columns[-3:])
train[shown].head(3)

,itching,skin_rash,nodal_skin_eruptions,continuous_sneezing,yellow_crust_ooze,prognosis,Unnamed: 133
0,1,1,1,0,0,Fungal infection,NaN
1,0,1,1,0,0,Fungal infection,NaN
2,1,0,1,0,0,Fungal infection,NaN


## 2. Problems in the raw files

These are the things the cleaning has to fix.

In [4]:
unnamed = [c for c in train.columns if c.startswith("Unnamed")]
print("Extra columns with no name:", unnamed)
for c in unnamed:
    print(f"  {c}: {train[c].isna().sum()} of {len(train)} values are empty")

duplicated = [c for c in train.columns if c.endswith(".1")]
print("\nDuplicated columns (pandas adds .1 to the second copy):", duplicated)
for c in duplicated:
    print(f"  {c} has the same values as {c[:-2]}: {train[c].equals(train[c[:-2]])}")

odd_columns = [c for c in train.columns if c != c.strip() or " " in c]
print(f"\nColumn names containing spaces: {len(odd_columns)}")
print("  examples:", odd_columns[:5])

all_names = pd.concat([train["prognosis"], test["prognosis"]]).dropna()
odd_diseases = sorted({p for p in all_names if p != p.strip()})
print("\nDisease names with spaces at the ends:", odd_diseases)

Extra columns with no name: ['Unnamed: 133']
  Unnamed: 133: 4920 of 4920 values are empty

Duplicated columns (pandas adds .1 to the second copy): ['fluid_overload.1']
  fluid_overload.1 has the same values as fluid_overload: False

Column names containing spaces: 4
  examples: ['spotting_ urination', 'foul_smell_of urine', 'dischromic _patches', 'Unnamed: 133']

Disease names with spaces at the ends: ['Diabetes ', 'Hypertension ']


## 3. Merge the two files

I'm not training a model, so I do not need a train/test split. So I put both files together to have all the cases.

In [5]:
df = pd.concat([train, test], ignore_index=True)
print(f"Merged: {df.shape[0]} rows ({len(train)} + {len(test)}), {df.shape[1]} columns")

Merged: 4962 rows (4920 + 42), 134 columns


## 4. Cleaning

Four steps, each with a before and after.

In [6]:
# Step 1: clean the column names (remove spaces at the ends, replace inner spaces with _)
old_names = list(df.columns)
df.columns = [c.strip().replace(" ", "_") for c in df.columns]
changed = [(o, n) for o, n in zip(old_names, df.columns) if o != n]
print(f"{len(changed)} column names changed. Examples:")
for o, n in changed[:5]:
    print(f"  {o!r} -> {n!r}")

4 column names changed. Examples:
  'spotting_ urination' -> 'spotting__urination'
  'foul_smell_of urine' -> 'foul_smell_of_urine'
  'dischromic _patches' -> 'dischromic__patches'
  'Unnamed: 133' -> 'Unnamed:_133'


In [7]:
# Step 2: drop the empty unnamed column and the duplicated column
before = df.shape[1]
df = df.loc[:, ~df.columns.str.startswith("Unnamed")]
df = df.drop(columns=["fluid_overload.1"], errors="ignore")
print(f"Columns: {before} -> {df.shape[1]}")

Columns: 134 -> 132


In [8]:
# Step 3: remove spaces at the ends of the disease names
before = df["prognosis"].nunique()
df["prognosis"] = df["prognosis"].str.strip()
print(f"Distinct disease names: {before} -> {df['prognosis'].nunique()}")

Distinct disease names: 41 -> 41


In [9]:
# Step 4: check that the symptom values are usable
symptom_columns = [c for c in df.columns if c != "prognosis"]
print("Symptom columns:", len(symptom_columns))
print("Missing values:", int(df.isna().sum().sum()))
print("All symptom values are 0 or 1:", bool(df[symptom_columns].isin([0, 1]).all().all()))

Symptom columns: 131
Missing values: 0
All symptom values are 0 or 1: True


## 5. Categories (written by hand)

The original data has no categories. I added them myself, by the main cause of each disease, so that the knowledge base can link each disease to a category (an InheritanceLink) and the engine can reason from a disease to its category.

- **ViralInfection:** Dengue, Chicken pox, Common Cold
- **BacterialInfection:** Typhoid, Tuberculosis, Pneumonia
- **ParasiticInfection:** Malaria

Pneumonia can be viral or bacterial. I chose bacterial. In the knowledge base the link Pneumonia to BacterialInfection has strength 0.7 instead of 1.0 because of this.

In [10]:
category_table = pd.DataFrame(sorted(CATEGORY.items(), key=lambda kv: (kv[1], kv[0])),
                              columns=["disease", "category"])
category_table

,disease,category
0,Pneumonia,BacterialInfection
1,Tuberculosis,BacterialInfection
2,Typhoid,BacterialInfection
3,Malaria,ParasiticInfection
4,Chicken pox,ViralInfection
5,Common Cold,ViralInfection
6,Dengue,ViralInfection


In [11]:
not_found = [d for d in CATEGORY if d not in set(df["prognosis"])]
print("Diseases from our list that are not in the data:", not_found)

Diseases from our list that are not in the data: []


## 6. The slice: 7 diseases, symptoms that appear

I kept only the 7 diseases, then kept only the symptoms that appear at least once in them. Then I added the category column.

In [12]:
df = df[df["prognosis"].isin(CATEGORY)].copy()
print(f"Rows kept: {len(df)}")
print()
print(df["prognosis"].value_counts().to_string())

symptoms = [c for c in df.columns if c != "prognosis"]
keep = [s for s in symptoms if df[s].sum() >= MIN_COUNT]
print(f"\nSymptoms before: {len(symptoms)}")
print(f"Symptoms kept (seen at least {MIN_COUNT} time in these diseases): {len(keep)}")
print(f"Symptoms dropped: {len(symptoms) - len(keep)}")

df["category"] = df["prognosis"].map(CATEGORY)
out = df[["prognosis", "category"] + keep]

Rows kept: 847

prognosis
Malaria         121
Chicken pox     121
Dengue          121
Typhoid         121
Tuberculosis    121
Common Cold     121
Pneumonia       121

Symptoms before: 131
Symptoms kept (seen at least 1 time in these diseases): 40
Symptoms dropped: 91


## 7. Result: `fever_slice.csv`

In [13]:
print(f"Shape: {out.shape[0]} rows, {out.shape[1]} columns (prognosis, category and {len(keep)} symptoms)")
out.groupby("prognosis").head(1).iloc[:, :8]

Shape: 847 rows, 42 columns (prognosis, category and 40 symptoms)


,prognosis,category,itching,skin_rash,continuous_sneezing,chills,joint_pain,vomiting
150,Malaria,ParasiticInfection,0,0,0,1,0,1
160,Chicken pox,ViralInfection,1,1,0,0,0,0
170,Dengue,ViralInfection,0,1,0,1,1,1
180,Typhoid,BacterialInfection,0,0,0,1,0,1
250,Tuberculosis,BacterialInfection,0,0,0,1,0,1
260,Common Cold,ViralInfection,0,0,1,1,0,0
270,Pneumonia,BacterialInfection,0,0,0,1,0,0


In [14]:
out.groupby("category")["prognosis"].agg(lambda s: ", ".join(sorted(s.unique()))).to_frame("diseases")

,diseases
category,
BacterialInfection,"Pneumonia, Tuberculosis, Typhoid"
ParasiticInfection,Malaria
ViralInfection,"Chicken pox, Common Cold, Dengue"


In [15]:
# Compare with the file already saved by prepare_data.py
if os.path.exists(EXISTING_SLICE):
    saved = pd.read_csv(EXISTING_SLICE)
    same_shape = saved.shape == out.shape
    same_columns = list(saved.columns) == list(out.columns)
    same_values = same_shape and same_columns and bool((saved.to_numpy() == out.to_numpy()).all())
    print(f"{EXISTING_SLICE} has the same shape: {same_shape}, same columns: {same_columns}, same values: {same_values}")
else:
    print(f"{EXISTING_SLICE} not found, nothing to compare.")

fever_slice.csv has the same shape: True, same columns: True, same values: True


## 8. From the slice to the knowledge base

Each disease has the same number of rows, n. For every disease and symptom:

- **strength** of the link Disease to Symptom = rows of the disease with the symptom / n
- **confidence** of the link = n / (n + k), with k = 10 (our choice)

A pair with no row gets no link. Below: how many links each disease gets, and its three strongest symptoms.

In [16]:
K = 10
n = out.groupby("prognosis").size()
counts = out.groupby("prognosis")[keep].sum()
strength = counts.div(n, axis=0)

links_per_disease = (counts > 0).sum(axis=1)
print("Disease -> Symptom links per disease:")
print(links_per_disease.to_string())
print(f"\nTotal Disease -> Symptom links: {int(links_per_disease.sum())}")

print("\nConfidence of the links, n / (n + k):")
print((n / (n + K)).round(4).to_string())

Disease -> Symptom links per disease:
prognosis
Chicken pox     11
Common Cold     17
Dengue          14
Malaria          8
Pneumonia       11
Tuberculosis    16
Typhoid         11

Total Disease -> Symptom links: 88

Confidence of the links, n / (n + k):
prognosis
Chicken pox     0.9237
Common Cold     0.9237
Dengue          0.9237
Malaria         0.9237
Pneumonia       0.9237
Tuberculosis    0.9237
Typhoid         0.9237


In [17]:
rows = []
for disease in strength.index:
    top = strength.loc[disease].sort_values(ascending=False, kind="stable").head(3)
    rows.append({"disease": disease,
                 "strongest symptoms (strength)": ", ".join(f"{s} {v:.2f}" for s, v in top.items())})
pd.DataFrame(rows)

,disease,strongest symptoms (strength)
0,Chicken pox,"malaise 1.00, red_spots_over_body 1.00, itchin..."
1,Common Cold,"phlegm 1.00, throat_irritation 1.00, redness_o..."
2,Dengue,"headache 1.00, nausea 1.00, loss_of_appetite 1.00"
3,Malaria,"muscle_pain 1.00, chills 0.95, vomiting 0.95"
4,Pneumonia,"chest_pain 1.00, fast_heart_rate 1.00, rusty_s..."
5,Tuberculosis,"loss_of_appetite 1.00, mild_fever 1.00, yellow..."
6,Typhoid,"chills 1.00, fatigue 1.00, high_fever 1.00"


## 9. What the data cannot show

These checks support the limitations of the project.

- **Repeated rows.** Each disease has 121 rows but only 8 to 10 distinct symptom patterns. If identical rows are independent patients, n = 121 is correct. If they are copies of the same record, n is larger than the number of independent cases, and the confidence values are too high. The file cannot tell us which is true.
- **Balanced classes.** Every disease has the same number of rows, so the data says nothing about how common each disease really is. The disease base rates are assumed, not measured.
- **Binary symptoms.** Each symptom is only present or absent, with no severity or timing.

In [18]:
unique_rows = out.drop_duplicates().groupby("prognosis").size()
table = pd.DataFrame({"rows": n, "unique rows": unique_rows})
table["duplicate rows"] = table["rows"] - table["unique rows"]
table

,rows,unique rows,duplicate rows
prognosis,,,
Chicken pox,121,10,111
Common Cold,121,9,112
Dengue,121,10,111
Malaria,121,8,113
Pneumonia,121,9,112
Tuberculosis,121,9,112
Typhoid,121,9,112


In [19]:
total_links = int((counts > 0).sum().sum())
links_at_one = int((strength == 1.0).sum().sum())
print(f"Links with strength exactly 1.0: {links_at_one} of {total_links}")
print(f"Rows per disease: min {n.min()}, max {n.max()} (balanced: {n.min() == n.max()})")

Links with strength exactly 1.0: 31 of 88
Rows per disease: min 121, max 121 (balanced: True)
